In [ ]:
# === Install dependencies ===
!pip install great_tables matplotlib numpy pandas polars pyarrow

In [ ]:
# === Imports ===
import great_tables as gt
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import sqlite3


In [ ]:
# === Load data ===
conn = sqlite3.connect("db.sqlite3")
df = pl.read_database("SELECT * FROM results", connection=conn)
conn.close()


In [ ]:
# === Filter by timeout ===
timeout = 600  # seconds
timeout_eps = 10  # security margin for slow solvers such as LPG
df = df.filter(pl.col("computation") <= timeout + timeout_eps)

In [ ]:
# === Filter the domains ===
# dom_filter_start = None
dom_filter_start = "socs2025-"
dom_filter_end = None
# dom_filter_end = "-num"
dom_exclude = []
# dom_exclude = ["match-cellar"]
# dom_exclude = ["counters", "drone"]

if dom_filter_start:
    df = df.filter(pl.col("problem").str.contains(dom_filter_start))
if dom_filter_end:
    df = df.filter(pl.col("problem").str.contains(dom_filter_end))



In [ ]:
# === Filter and set an order for the planners on the table ===
planners_order = [
    "optic",
    "aries-optic",
    "lpg",
    "aries-lpg",
    "aries",
    # "aries-activitybool",
    # "aries-causal",
    # "aries-nofact",
    # "aries-sym"
]

all_planners = df.select(pl.col("planner").unique().sort()).to_series().to_list()
print(all_planners)
planners_order = [p for p in planners_order if p in all_planners]
print(planners_order)
# planners_order = all_planners
df = df.filter(pl.col("planner").is_in(planners_order))


In [ ]:
# === Metrics Configuration ===
metrics = ["Cov", "IPC"] #, "Agl"]

In [ ]:
# === VBS Configuration ===
vbs_configs = [
    (["optic", "lpg"], "VBS OL"),
    (["optic", "lpg", "aries"], "VBS OLA")
]


In [ ]:
# === Delta Configuration ===
delta_configs = [
    ("VBS OL", "VBS OLA", "Δ(+aries)"),
]

In [ ]:
# === Preprocessing: add domain columns ===
df = df.with_columns([pl.col("problem").str.split(":").list.get(0).alias("domain")])
if dom_filter_start:
    df = df.with_columns([pl.col("domain").str.replace(dom_filter_start, "").alias("domain")])
if dom_filter_end:
    df = df.with_columns([pl.col("domain").str.replace(dom_filter_end, "").alias("domain")])
if dom_exclude:
    df = df.filter(~pl.col("domain").is_in(dom_exclude))
print(df.select(pl.col('domain').unique()).to_series().to_list())


In [ ]:
# === Preprocessing: add number of instances column ===
num_instances = (
    df.group_by("domain")
      .agg(pl.col("problem").n_unique().alias("N_instances"))
)
df = df.join(num_instances, on="domain", how="left")

In [ ]:
# === Modify computation for aries-optic and aries-lpg ===
# Create a mapping to get base planner results
base_planners = {"aries-optic": "optic", "aries-lpg": "lpg"}

# Process each aries planner
for aries_planner, base_planner in base_planners.items():
    # Get all problems for this aries planner
    aries_data = df.filter(pl.col("planner") == aries_planner)
    base_data = df.filter(pl.col("planner") == base_planner)
    
    # Join to get both computations
    combined_data = aries_data.join(
        base_data.select(["problem", "computation"]).rename({"computation": "base_computation"}),
        on="problem",
        how="left"
    )
    
    # Calculate sum of computations
    combined_data = combined_data.with_columns([
        (pl.col("computation") + pl.col("base_computation")).alias("computation")
    ]).drop("base_computation")
    
    # Store original problems for this planner
    original_problems = set(aries_data["problem"].to_list())
    
    # Filter by computation < 610
    filtered_data = combined_data.filter(pl.col("computation") < 610)
    filtered_problems = set(filtered_data["problem"].to_list())
    
    # Find problems that were removed by filtering
    removed_problems = original_problems - filtered_problems
    
    # For removed problems, use ONESHOT solution of the base solver
    if removed_problems:
        oneshot_replacements = []
        for problem in removed_problems:
            # Find the ONESHOT result for the base planner
            oneshot_result = df.filter(
                (pl.col("planner") == base_planner) & 
                (pl.col("problem") == problem) & 
                (pl.col("mode") == "ONESHOT")
            )
            if not oneshot_result.is_empty():
                # Create replacement entry with aries planner name
                replacement = oneshot_result.with_columns(pl.lit(aries_planner).alias("planner"))
                oneshot_replacements.append(replacement)
        
        # Add ONESHOT replacements to filtered data
        if oneshot_replacements:
            oneshot_df = pl.concat(oneshot_replacements)
            filtered_data = pl.concat([filtered_data, oneshot_df])
    
    # Update df by removing old entries and adding new ones
    df = df.filter(pl.col("planner") != aries_planner)
    df = pl.concat([df, filtered_data])

print(f"Modified computation for aries-optic and aries-lpg planners")

In [ ]:
# === Preprocessing: fill the missing problems from warm-up ===
# There are some issues with the warm-up solvers, if the solver times out,
# there is no entry in the database. In this case, we fill the missing
# problems with the warming solver's ONESHOT result, e.g., use the ONESHOT
# result of lpg for aries-lpg.

all_problems = df.select(pl.col("problem").unique().sort()).to_series().to_list()
for planner in planners_order:
    if not planner.startswith("aries-"):
        continue
    for problem in all_problems:
        if not df.filter(
            (pl.col("planner") == planner) & (pl.col("problem") == problem) & (pl.col("status") != "TIMEOUT")
        ).is_empty():
            continue
        # Find the ONESHOT result for the same problem
        oneshot = df.filter(
            (pl.col("planner") == planner.split("-")[1])
            & (pl.col("problem") == problem)
            & (pl.col("mode") == "ONESHOT")
        )
        if oneshot.is_empty():
            continue
        # Fill the missing entry
        df = df.vstack(oneshot.with_columns(pl.lit(planner).alias("planner")))

In [ ]:
# === Best quality per problem (for IPC) ===
best_quality = (
  df.with_columns([
    pl.when(pl.col("status") != "SOLVED")
      .then(float("inf"))
      .otherwise(pl.col("quality"))
      .alias("quality_for_best")
  ])
  .group_by("problem")
  .agg(
    pl.col("quality_for_best")
      .fill_nan(float("inf"))
      .fill_null(float("inf"))
      .min()
      .alias("best_quality"),
  )
)

df = df.join(best_quality, on="problem", how="left")


In [ ]:
# === Per (planner, problem): take the best result ===
best_quality_solved = (
    df.filter(pl.col("status") == "SOLVED")
      .sort(["planner", "problem", "quality"])
      .unique(subset=["planner", "problem"], keep="first")
)
best_computation_solved = (
    df.filter(pl.col("status") == "SOLVED")
      .sort(["planner", "problem", "computation"])
      .unique(subset=["planner", "problem"], keep="first")
      .select(["planner", "problem", "computation"])
      .rename({"computation": "best_computation"})
)

solved = (
    best_quality_solved.join(
        best_computation_solved, on=["planner", "problem"], how="left"
    )
    .with_columns([pl.col("best_computation").alias("computation")])
    .drop("best_computation")
)

solved_keys = set((row[0], row[1]) for row in zip(solved["planner"].to_list(), solved["problem"].to_list()))
all_keys = set((row[0], row[1]) for row in zip(df["planner"].to_list(), df["problem"].to_list()))
unsolved_keys = all_keys - solved_keys

if unsolved_keys:
    unsolved = (
        # Select the Oneshot result per default
        pl.DataFrame(list(unsolved_keys), schema=["planner", "problem"], orient="row")
            .join(df, on=["planner", "problem"], how="left")
            .sort("mode", descending=True) # "ONESHOT" < "ANYTIME"
            .unique(subset=["planner", "problem"], keep="first")
            .select(solved.columns)
    )
    combo = pl.concat([solved, unsolved])
else:
    combo = solved


# Ensure there is on entry per (planner, domain)
missing_entries = (
    combo.group_by("planner", "domain", "N_instances")
         .len()
         .filter(pl.col("len") != pl.col("N_instances"))
)
print(missing_entries)

In [ ]:
# === VBS Computation ===

def compute_vbs_data(combo_df, planners, vbs_name):
    """Compute VBS data by selecting best result per problem from specified planners"""
    vbs_rows = []
    problems = combo_df.select(pl.col("problem").unique()).to_series().to_list()
    
    for problem in problems:
        problem_data = combo_df.filter(
            (pl.col("problem") == problem) & 
            (pl.col("planner").is_in(planners))
        )
        
        if problem_data.is_empty():
            continue
        
        # Filter by solved status using the status column (not the computed solved column)
        solved_data = problem_data.filter(pl.col("status") == "SOLVED")
        
        if solved_data.is_empty():
            # No solver solved this problem, take first available (all failed)
            best_result = problem_data.slice(0, 1)
            best_result = best_result.with_columns([
                pl.lit(vbs_name).alias("planner"),
                pl.lit("UNSOLVED").alias("status")
            ])
        else:
            # Select best quality among solved results
            best_result = solved_data.sort("quality").slice(0, 1)
            best_result = best_result.with_columns([
                pl.lit(vbs_name).alias("planner")
            ])
        
        vbs_rows.append(best_result)
    
    if vbs_rows:
        return pl.concat(vbs_rows)
    else:
        return combo_df.head(0).with_columns(pl.lit(vbs_name).alias("planner"))

# Compute all VBS data using the combo dataframe (without metrics computed yet)
vbs_dataframes = []
vbs_names = []
for planners, vbs_name in vbs_configs:
    vbs_df = compute_vbs_data(combo, planners, vbs_name)
    vbs_dataframes.append(vbs_df)
    vbs_names.append(vbs_name)

# Add VBS data to combo
if vbs_dataframes:
    combo_with_vbs = pl.concat([combo] + vbs_dataframes)
else:
    combo_with_vbs = combo

# Update planners order to include VBS
planners_order_with_vbs = planners_order + vbs_names

# Now compute metrics on the combo_with_vbs dataframe (includes VBS data)
# Add solved column
combo_with_vbs = combo_with_vbs.with_columns([
    (pl.col("status") == "SOLVED").alias("solved")
])

# IPC calculation
def ipc_func(row):
    s, q, bq = row["solved"], row["quality"], row["best_quality"]
    if s and (q is not None):
        if q == bq:
            return 1.0
        return bq / q
    return 0.0

combo_with_vbs = combo_with_vbs.with_columns([
    pl.struct(["solved", "quality", "best_quality"])
    .map_elements(ipc_func, return_dtype=pl.Float64)
    .alias("ipc")
])

# Compute aggregation with VBS
agg_with_vbs = (
    combo_with_vbs.group_by(["domain", "planner", "N_instances"])
    .agg([
        pl.col("solved").sum().alias("N_solved"),
        pl.col("ipc").mean().alias("IPC"),
    ])
    .with_columns([
        (100 * pl.col("N_solved") / pl.col("N_instances")).round(2).alias("Cov"),
        (100 * pl.col("IPC")).round(2).alias("IPC"),
    ])
    .drop(["N_solved"])
    .sort(["domain", "planner"])
    .select(["domain", "planner", "N_instances"] + metrics)
)

print(f"VBS planners computed: {vbs_names}")
print(f"Final planners order: {planners_order_with_vbs}")

In [ ]:
# === Delta Computation ===

delta_results = {}
domains = agg_with_vbs["domain"].unique().to_list()

for baseline_planner, comparison_planner, delta_label in delta_configs:
    delta_for_this_config = []
    
    for domain in domains:
        domain_data = agg_with_vbs.filter(pl.col("domain") == domain)
        domain_size = domain_data.row(0)[2]
        
        # Get baseline and comparison values
        baseline_data = domain_data.filter(pl.col("planner") == baseline_planner)
        comparison_data = domain_data.filter(pl.col("planner") == comparison_planner)
        
        if baseline_data.height > 0 and comparison_data.height > 0:
            baseline_cov = baseline_data.row(0)[3]  # Coverage
            baseline_ipc = baseline_data.row(0)[4]  # IPC
            comparison_cov = comparison_data.row(0)[3]  # Coverage
            comparison_ipc = comparison_data.row(0)[4]  # IPC
            
            delta_cov = round(comparison_cov - baseline_cov, 2)
            delta_ipc = round(comparison_ipc - baseline_ipc, 2)
        else:
            delta_cov = np.nan
            delta_ipc = np.nan
        
        delta_for_this_config.append([domain, domain_size, delta_cov, delta_ipc])
    
    # Store delta results for this configuration
    delta_results[delta_label] = pl.DataFrame(
        delta_for_this_config, 
        schema=["domain", "N_instances", "Cov", "IPC"], 
        orient="row"
    )

print(f"Delta configurations computed: {list(delta_results.keys())}")

In [ ]:
# === Metrics computation ===
from math import log10

# Coverage
if "Cov" in metrics:
    combo = combo.with_columns([
        (pl.col("status") == "SOLVED").alias("solved")
    ])

# IPC
def ipc_func(row):
    s, q, bq = row["solved"], row["quality"], row["best_quality"]
    if s and (q is not None):
        if q == bq:
            return 1.0
        return bq / q
    return 0.0

if "IPC" in metrics:
    combo = combo.with_columns([
        pl.struct(["solved", "quality", "best_quality"])
        .map_elements(ipc_func, return_dtype=pl.Float64)
        .alias("ipc")
    ])

# Agile
def agl_func(row):
    s, c, t = row["solved"], row["computation"], row["timeout"]
    if not s or c is None or c > t:
        return 0.0
    if c < 1:
        return 1.0
    return 1 - log10(c) / log10(t)

if "Agl" in metrics:
    combo = combo.with_columns([
        pl.struct(["solved", "computation", "timeout"])
        .map_elements(agl_func, return_dtype=pl.Float64)
        .alias("agl")
    ])

# Compute base aggregation for original planners
agg = (
    combo.group_by(["domain", "planner", "N_instances"])
    .agg([
        pl.col("solved").sum().alias("N_solved"),
        pl.col("ipc").mean().alias("IPC"),
    ])
    .with_columns([
        (100 * pl.col("N_solved") / pl.col("N_instances")).round(2).alias("Cov"),
        (100 * pl.col("IPC")).round(2).alias("IPC"),
    ])
    .drop(["N_solved"])
    .sort(["domain", "planner"])
    .select(["domain", "planner", "N_instances"] + metrics)
)

In [ ]:
# === Table Generation ===

# Core of the table with VBS and Delta
results = []
for domain in domains:
    domain_size = agg_with_vbs.filter(pl.col("domain") == domain).row(0)[2]
    row = [domain, domain_size]
    
    # Add all planners (original + VBS)
    for planner in planners_order_with_vbs:
        subdf = agg_with_vbs.filter((pl.col("domain") == domain) & (pl.col("planner") == planner))
        if subdf.height > 0:
            r = subdf.row(0)
            row += [r[3], r[4]]  # Cov, IPC
        else:
            row += [np.nan, np.nan]
    
    # Add Delta values for each configured delta
    for _, _, delta_label in delta_configs:
        delta_row = delta_results[delta_label].filter(pl.col("domain") == domain)
        if delta_row.height > 0:
            d = delta_row.row(0)
            row += [d[2], d[3]]  # Delta Cov, Delta IPC
        else:
            row += [np.nan, np.nan]
    
    results.append(row)

# Build final table schema
delta_labels = [delta_label for _, _, delta_label in delta_configs]
planners_order_final = planners_order_with_vbs + delta_labels
metric_cols = [f"{p}_{m}" for p in planners_order_final for m in metrics]
table_df = pl.DataFrame(results, schema=["Domains", "Size"] + metric_cols, orient="row")
table_df = table_df.sort("Domains")

# Average row at the end
average_row = ["Average", table_df["Size"].sum()]
for planner in planners_order_final:
    for metric in metrics:
        vals = table_df[f"{planner}_{metric}"].to_numpy()
        vals = vals[~np.isnan(vals)]
        if len(vals) > 0:
            average_row.append(np.round(np.mean(vals), 2))
        else:
            average_row.append(np.nan)
table_df = table_df.vstack(pl.DataFrame([average_row], schema=table_df.columns, orient="row"))

# Format the table
tbl = gt.GT(table_df)
for planner in planners_order_final:
    tbl = tbl.tab_spanner(label=planner, columns=[f"{planner}_{m}" for m in metrics])
tbl = tbl.cols_label(**{mc: mc.split("_")[1] for mc in metric_cols})

# Add coloration - only for original planners (exclude VBS and Delta)
for i, domain in enumerate(table_df["Domains"]):
    for metric in metrics:
        # Only consider original planners for coloration
        vals = [table_df[f"{planner}_{metric}"][i] for planner in planners_order]
        if all(np.isnan(vals)):
            continue
        max_val = np.nanmax(vals)
        min_val = np.nanmin(vals)
        # Only color the original planners
        for planner in planners_order:
            col = f"{planner}_{metric}"
            if table_df[col][i] in [max_val, min_val]:
                color="blue" if table_df[col][i] == max_val else "red"
                tbl = tbl.tab_style(
                    style=gt.style.text(
                        weight="bold",
                        color=color,
                    ),
                    locations=gt.loc.body(
                        columns=col, 
                        rows=i
                    )
                )

tbl

In [ ]:
import numpy as np

def generate_research_latex_table(table_df, planners_order, metrics):
    """
    Generate LaTeX table matching the research paper format exactly
    """
    
    latex_lines = []
    
    # Table setup with exact formatting
    latex_lines.append("\\begin{table}[t]")
    latex_lines.append("    \\centering")
    latex_lines.append("    \\tiny")
    latex_lines.append("    \\renewcommand{\\arraystretch}{1.35}")
    latex_lines.append("    \\def\\hs{\\hspace{0.1cm}}")
    
    # Build column specification - exactly matching the research format
    # For 3 planners: domain + separator + planner1 + sep + planner2 + sep + planner3
    col_spec = "@{\\hs}r@{\\hs}r@{}||@{}r@{\\hs}r@{\\hs}r@{\\hs}|@{}r@{\\hs}r@{\\hs}r@{\\hs}|@{}r@{\\hs}r@{\\hs}r@{\\hs}"
    
    latex_lines.append(f"    \\begin{{tabular}}{{{col_spec}}}")
    latex_lines.append("        \\toprule")
    
    # Create header row with planner names
    # Pattern: domain & separator & separator & multicolumn{2}{planner1} & separator & multicolumn{2}{planner2} & separator & multicolumn{2}{planner3}
    header_parts = ["\\multicolumn{1}{c}{}", "", ""]  # Domain + 2 separators
    cmidrule_parts = []
    
    col_idx = 4  # Start after domain + 2 separators
    for i, planner in enumerate(planners_order):
        # Add planner name with multicolumn spanning 2 columns
        planner_cmd = f"\\{planner.lower()}"  # Assumes you have \optic, \lpg, \aries commands
        header_parts.append(f"\\multicolumn{{2}}{{c}}{{{planner_cmd}}}")
        
        # Add cmidrule for this planner's 2 columns
        cmidrule_parts.append(f"\\cmidrule{{{col_idx}-{col_idx+1}}}")
        col_idx += 2  # Move past the 2 metric columns
        
        # Add separator after planner (except for last planner)
        if i < len(planners_order) - 1:
            header_parts.append("")
            col_idx += 1  # Move past separator
    
    header_line = " & ".join(header_parts) + " \\\\"
    cmidrule_line = "".join(cmidrule_parts)
    
    latex_lines.append(f"        {header_line}{cmidrule_line}")
    
    # Create subheader with metric names
    subheader_parts = ["\\multicolumn{1}{c}{}", "", ""]  # Domain + 2 separators
    
    for i, planner in enumerate(planners_order):
        # Add metric columns
        for metric in metrics:
            subheader_parts.append(f"\\multicolumn{{1}}{{c}}{{{metric}}}")
        
        # Add separator after planner (except for last planner)
        if i < len(planners_order) - 1:
            subheader_parts.append("")
    
    subheader_line = " & ".join(subheader_parts) + " \\\\"
    latex_lines.append(f"        {subheader_line}\\midrule")
    
    # Prepare data for finding best/worst values for formatting
    def find_best_worst_values(df, planners, metrics):
        """Find best and worst values for each metric across planners"""
        best_worst = {}
        
        # Exclude average row for comparison
        data_rows = df[:-1]
        
        for row_data in data_rows.iter_rows():
            domain = row_data[0]
            best_worst[domain] = {}
            
            for metric_idx, metric in enumerate(metrics):
                values = []
                positions = []
                
                for planner_idx, planner in enumerate(planners):
                    col_idx = 2 + planner_idx * len(metrics) + metric_idx
                    val = row_data[col_idx]
                    if val is not None and not (isinstance(val, float) and np.isnan(val)):
                        values.append(val)
                        positions.append((planner_idx, metric_idx))
                
                if values:
                    max_val = max(values)
                    min_val = min(values)
                    best_worst[domain][metric] = {
                        'max': max_val,
                        'min': min_val,
                        'positions': positions,
                        'values': values
                    }
        
        return best_worst
    
    best_worst = find_best_worst_values(table_df, planners_order, metrics)
    
    # Add data rows (excluding average)
    data_rows = table_df[:-1]
    
    for row in data_rows.iter_rows():
        domain = row[0].capitalize()
        size = row[1]
        
        # Start with domain name in braces and separators
        line_parts = [f"{{{domain} ({size})}}", "", ""]
        
        # Add data for each planner
        for planner_idx, planner in enumerate(planners_order):
            # Add metric values
            for metric_idx, metric in enumerate(metrics):
                col_idx = 2 + planner_idx * len(metrics) + metric_idx
                val = row[col_idx]
                
                if val is None or (isinstance(val, float) and np.isnan(val)):
                    formatted_val = "---"
                else:
                    formatted_val = f"{val:.2f}"
                    
                    # Check if this is a best value and should be bold
                    if domain in best_worst and metric in best_worst[domain]:
                        if val == best_worst[domain][metric]['max']:
                            formatted_val = f"{{\\textbf{{{formatted_val}}}}}"
                
                line_parts.append(f"{{{formatted_val}}}")
            
            # Add separator after planner (except last)
            if planner_idx < len(planners_order) - 1:
                line_parts.append("")
        
        line = " & ".join(line_parts) + " \\\\"
        latex_lines.append(f"        {line}")
    
    # Add dashed line before average
    total_cols = 3 + len(planners_order) * 2 + (len(planners_order) - 1)  # 3 initial + 2 per planner + separators
    latex_lines.append(f"        \\cdashline{{1-{total_cols}}}")
    
    # Add average row
    avg_row = list(table_df[-1:].iter_rows())[0]  # Get the last row properly
    avg_parts = [f"{{{avg_row[0]}}}", "", ""]  # Domain + 2 separators
    
    # Find best average values
    avg_values = {}
    for metric_idx, metric in enumerate(metrics):
        values = []
        for planner_idx, planner in enumerate(planners_order):
            col_idx = 2 + planner_idx * len(metrics) + metric_idx
            val = avg_row[col_idx]
            if val is not None and not (isinstance(val, float) and np.isnan(val)):
                values.append(val)
        if values:
            avg_values[metric] = max(values)
    
    for planner_idx, planner in enumerate(planners_order):
        # Add metric values
        for metric_idx, metric in enumerate(metrics):
            col_idx = 2 + planner_idx * len(metrics) + metric_idx
            val = avg_row[col_idx]
            
            if val is None or (isinstance(val, float) and np.isnan(val)):
                formatted_val = "---"
            else:
                formatted_val = f"{val:.2f}"
                
                # Bold if best average
                if metric in avg_values and val == avg_values[metric]:
                    formatted_val = f"{{\\textbf{{{formatted_val}}}}}"
            
            avg_parts.append(f"{{{formatted_val}}}")
        
        # Add separator after planner (except last)
        if planner_idx < len(planners_order) - 1:
            avg_parts.append("")
    
    avg_line = " & ".join(avg_parts) + " \\\\"
    latex_lines.append(f"        {avg_line}\\bottomrule")
    
    # Close table
    latex_lines.append("    \\end{tabular}")
    latex_lines.append("    \\caption{Comparison of the different planners on the different domains.}")
    latex_lines.append("    \\label{tab:metrics}")
    latex_lines.append("\\end{table}")
    
    return "\n".join(latex_lines)

# Generate the research-style LaTeX table
research_latex = generate_research_latex_table(table_df, planners_order, metrics)
print(research_latex)

In [ ]:
# === Cactus plot ===
combo_pd = combo.select(["planner", "problem", "solved", "computation"]).to_pandas()

plt.figure(figsize=(10, 6))
for planner in combo_pd["planner"].unique():
    if planner in ["aries-lpg", "aries-optic"]:
        continue
    times = combo_pd[(combo_pd["planner"] == planner) & (combo_pd["solved"])]["computation"].dropna()
    times = np.sort(times.values)
    y = times
    x = np.arange(1, len(y) + 1)
    plt.plot(y, x, label=planner)

plt.ylabel("Number of Problems Solved")
plt.xlabel("Computation Time (s)")
plt.title("Cactus Plot (Planning Time per Planner)")
plt.grid(True, which="both", linestyle="--", alpha=0.5)
plt.legend(title="Planner")
plt.tight_layout()
plt.xscale("log")
plt.show()